In [ ]:
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import scipy as sp
import sympy as sym
from tqdm.auto import tqdm

sys.path.append( '../src' )
from GaussianQFINumerics import FIM, vectorized_QFIM  #,SLD,_SLD_precomp,FIE
#%matplotlib widget

In [ ]:
r,nbar, η1,η2,α, r2 = sym.symbols('r nbar eta1 eta2 alpha r2')

In [ ]:
σ0 = np.matrix([
        [1+2*sym.sinh(r)**2,0,0,0,-2*sym.sinh(r)*sym.cosh(r),0 ],
        [0,1+2*sym.sinh(r)**2,0,0,0,2*sym.sinh(r)*sym.cosh(r)],
        [0,0,1,0,0,0],
        [0,0,0,1,0,0],
        [-2*sym.sinh(r)*sym.cosh(r),0,0,0,1+2*sym.sinh(r)**2,0],
        [0,2*sym.sinh(r)*sym.cosh(r),0,0,0, 1+2*sym.sinh(r)**2]
    ])

In [ ]:
print(σ0)

In [ ]:
X1 = np.matrix([
        [α,0,sym.sqrt(1-α**2),0,0,0],
        [0,α,0,sym.sqrt(1-α**2),0,0],
        [-sym.sqrt(1-α**2),0,α,0,0,0],
        [0,-sym.sqrt(1-α**2),0,α,0,0],
        [0,0,0,0,1,0],
        [0,0,0,0,0,1]
    ])

In [ ]:
print(X1)

In [ ]:
X2 = np.matrix([
        [1,0,0,0,0,0],
        [0,1,0,0,0,0],
        [0,0,sym.cosh(r2),0,sym.sinh(r2),0],
        [0,0,0,sym.cosh(r2),0,-sym.sinh(r2)],
        [0,0,sym.sinh(r2),0,sym.cosh(r2),0],
        [0,0,0,-sym.sinh(r2),0,sym.cosh(r2)]
    ])

In [ ]:
X2

In [ ]:
X3 = np.matrix([
    [sym.sqrt(η1),0,0,0,0,0],
    [0,sym.sqrt(η1),0,0,0,0],
    [0,0,sym.sqrt(η2),0,0,0],
    [0,0,0,sym.sqrt(η2),0,0],
    [0,0,0,0,1,0],
    [0,0,0,0,0,1]
    ])

In [ ]:
X3

In [ ]:
Y3 = np.matrix([
    [(1-η1) + 2*nbar,0,0,0,0,0],
    [0,(1-η1) + 2*nbar,0,0,0,0],
    [0,0,(1-η2) + 2*nbar,0,0,0],
    [0,0,0,(1-η2) + 2*nbar,0,0],
    [0,0,0,0,0,0],
    [0,0,0,0,0,0]
    ])

In [ ]:
Y3

In [ ]:
X = X3@X2@X1

In [ ]:
print(X)

In [ ]:
X2@X1

In [ ]:
X1

In [ ]:
σ3 = sym.simplify(X@σ0@np.transpose(X) + Y3)

In [ ]:
dσ3dη1 = sym.diff(σ3,η1)

In [ ]:
dσ3dη2 = sym.diff(σ3,η2)

In [ ]:
dσ3dη1

In [ ]:
σ3

In [ ]:
σ = sym.lambdify((r,nbar, η1,η2,α,r2),σ3)

In [ ]:
dσs = [sym.lambdify((r,nbar, η1,η2,α,r2),dσ3dη1),sym.lambdify((r,nbar, η1,η2,α,r2),dσ3dη2)]

In [ ]:
def vec_QFIEs(QFIMs,a):
    # Currently only written for 2 parameter estimation
    aeff = a #if (np.abs(a) > .5) else (-a+ np.sign(a))
    B = [[1/aeff,0],[-np.sqrt(1-aeff**2)/aeff,1]]
    shape = QFIMs.shape[0:-2]
    QFIEs = np.zeros(shape)
    #postmult = np.nan_to_num(B@QFIMs@np.transpose(B),posinf=0)
    #Qtm1 = sp.linalg.pinv(postmult,rtol=1e-50)
    for arg in tqdm(np.ndindex(shape),total=np.prod(shape),smoothing=.01):
        Qtm1 = sp.linalg.pinvh(np.nan_to_num(B@QFIMs[*arg]@np.transpose(B),posinf=0),rtol=1e-120)#Specifiy tolerance since this has had issue otherwise
        FIEval = 1/Qtm1[0,0]
        QFIEs[*arg] = FIEval
    return np.nan_to_num(QFIEs,posinf=0)

In [ ]:
η1vals = np.linspace(.01,.4,40)
η2vals = np.linspace(.01,.4,40)
αvals = np.linspace(-.99,.99,160)
rvals = [1e-7,np.arcsinh(np.sqrt(2)*np.sinh(1e-7))]
r2vals = [1e-7,0]
nbarvals = [.01]
#arggrid = np.meshgrid(rvals,nbarvals,η1vals,η2vals,αvals,r2vals,indexing='ij')

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    QFIMs = np.real_if_close(vectorized_QFIM(σ,dσs,rvals,nbarvals,η1vals,η2vals,αvals,r2vals))

In [ ]:
#[η1s, η2s, αvals, rvals, QFIMs] = load_state('../_data/QFIEs3.npz')
target = np.sqrt(9/16)
QFIEs = vec_QFIEs(QFIMs,target)
#del QFIMs

In [ ]:
targeteddata1 = QFIEs[0,0,:,:,:,0]
targeteddata2 = QFIEs[1,0,:,:,:,1]
maximized1 = sp.signal.wiener(np.nanmax(targeteddata1,axis=2))
maximized2 = sp.signal.wiener(np.nanmax(targeteddata2,axis=2))

[η1grid,η2grid] = np.meshgrid(η1vals,η2vals,indexing='ij')
select_not_huge = (η1grid<.8)*(η2grid<.8)
fig = plt.figure()
ax = fig.add_subplot(projection='3d')
ax.plot_trisurf(η1grid[select_not_huge],η2grid[select_not_huge],maximized1[select_not_huge])

ax.plot_trisurf(η1grid[select_not_huge],η2grid[select_not_huge],maximized2[select_not_huge])
ax.set_xlabel(r'$\eta_1$')
ax.set_ylabel(r'$\eta_2$')
ax.set_zlabel(r'$\mathcal{F}$')
ax.set_box_aspect(None, zoom=0.8)
plt.show()

In [ ]:
diff = maximized1-maximized2#sp.signal.wiener(maximized1,(10,10))-sp.signal.wiener(maximized2,(10,10))
fig = plt.figure()
ax = fig.add_subplot(projection='3d')
ax.plot_trisurf(η1grid[select_not_huge],η2grid[select_not_huge],diff[select_not_huge])
ax.set_xlabel(r'$\eta_1$')
ax.set_ylabel(r'$\eta_2$')
ax.set_zlabel(r'$\mathcal{F}_1-\mathcal{F}_2$')
ax.set_box_aspect(None, zoom=0.8)
plt.show()

In [ ]:
rel = 100*sp.signal.wiener(diff/maximized2)
fig = plt.figure()
ax = fig.add_subplot(projection='3d')
ax.plot_trisurf(η1grid[select_not_huge],η2grid[select_not_huge],rel[select_not_huge])
ax.set_xlabel(r'$\eta_1$')
ax.set_ylabel(r'$\eta_2$')
ax.set_zlabel(r'% advantage')
ax.set_box_aspect(None, zoom=0.8)
plt.show()

In [ ]:
αmax = αvals[np.argmax(targeteddata1,axis=2)]
fig = plt.figure()
ax = fig.add_subplot(projection='3d')
ax.plot_surface(η1grid,η2grid,np.abs(αmax))
ax.set_xlabel(r'$\eta_1$')
ax.set_ylabel(r'$\eta_2$')
ax.set_zlabel(r'$\alpha$')
ax.set_box_aspect(None, zoom=0.8)
plt.show()

In [ ]:
#r2max = np.array(r2vals)[np.argmax(np.nanmax(targeteddata,axis=2),axis=2)]
#fig = plt.figure()
#ax = fig.add_subplot(projection='3d')
#ax.plot_surface(η1grid,η2grid,np.log(r2max))
#plt.show()

In [ ]:
#selector = r2max >0
#fig = plt.figure()
#ax = fig.add_subplot(projection='3d')
#ax.plot_trisurf(η1grid[selector],η2grid[selector],np.abs(αmax[selector]))
#plt.show()

In [ ]:
σ3.subs(r2,0)

In [ ]:
(X2@X2)

In [ ]:
η1vals

In [ ]:
η2val =np.argmax(np.max(maximized2,axis=0))

In [ ]:
η1val = np.argmax(np.max(maximized2,axis=1))

In [ ]:
αval = np.argmax(QFIEs[1,0,η1val,η2val,:,1])

In [ ]:
σbad = σ(rvals[1],nbarvals[0],η1vals[η1val],η2vals[η2val],αvals[αval],r2vals[1])

In [ ]:
σbad

In [ ]:
dσsbad = [dσ(rvals[1],nbarvals[0],η1vals[η1val],η2vals[η2val],αvals[αval],r2vals[1]) for dσ in dσs]

In [ ]:
FIMbad = FIM(σbad,dσsbad)

In [ ]:
vec_QFIEs(FIMbad,target)

In [ ]:
FIMbad

In [ ]:
(rvals[1],nbarvals[0],η1vals[η1val],η2vals[η2val],αvals[αval],r2vals[1])

In [ ]:
sp.linalg.pinvh([[4.60203408e-21, 2.62928100e-17],
 [2.62928100e-17, 1.50218762e-13]],rtol=1e-30)

In [ ]:
sp.linalg.inv([[4.60203408e-21, 2.62928100e-17],
 [2.62928100e-17, 1.50218762e-13]])

In [ ]:
B = [[1/target,0],[-np.sqrt(1-target**2)/target,1]]

In [ ]:
toinv = B@FIMbad@np.transpose(B)

In [ ]:
sp.linalg.inv(toinv)

In [ ]:
sp.linalg.pinvh(toinv)

In [ ]:
select_not_huge.shape

In [ ]:
maximized2[select_not_huge].shape

In [ ]:
maximized2.shape

In [ ]:
np.sum(select_not_huge)